# Import

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-upstage openai
!pip cache purge

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
data_path = "YOUR_PATH"

req_path = os.path.join(data_path, 'requirements.txt')
!pip install -r "{req_path}"

In [ ]:
import os
import re
import time
import pickle
import json
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import html2text
import wikipediaapi
from langchain.schema import Document
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from openai import OpenAI
from datasets import load_dataset

In [ ]:
# @title 0. Configuration

# API key 설정
load_dotenv()

UPSTAGE_API_KEY = os.getenv('UPSTAGE_API_KEY')

if not UPSTAGE_API_KEY:
    raise ValueError("UPSTAGE_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요.")


embedding_save_path = os.path.join(data_path, "embeddings")
os.makedirs(embedding_save_path, exist_ok=True)

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

# 1. Parsing+Embedding: Wikipedia, SEP crawling

In [ ]:
# @title 1-1: MMLU-Pro 데이터셋 분석

def analyze_mmlu_pro_categories():
    """MMLU-Pro 5개 카테고리의 주요 토픽 분석"""

    category_insights = {
        "law": {
            "topics": [
                "Contract law", "Tort law", "Constitutional law", "Criminal law",
                "Property law", "Administrative law", "International law",
                "Labor law", "Environmental law", "Family law", "Corporate law",
                "Evidence law", "Civil procedure", "Legal reasoning", "Jurisprudence"
            ],
            "keywords": ["liability", "statute", "plaintiff", "defendant", "verdict"],
            "question_types": ["case analysis", "legal principle application"]
        },
        "psychology": {
            "topics": [
                "Cognitive psychology", "Behavioral psychology", "Developmental psychology",
                "Social psychology", "Clinical psychology", "Abnormal psychology",
                "Learning theory", "Memory", "Cognition", "Motivation", "Emotion",
                "Personality theory", "Group dynamics", "Psychopathology",
                "Therapeutic approaches", "Statistics and research design"
            ],
            "keywords": ["behavior", "stimulus", "response", "cognition", "neurotransmitter"],
            "question_types": ["theory application", "experimental interpretation"]
        },
        "business": {
            "topics": [
                "Business strategy", "Marketing management", "Organizational behavior",
                "Financial management", "Accounting", "Strategic planning",
                "Management theory", "Organizational structure", "Human resources",
                "Business ethics", "Supply chain", "Entrepreneurship",
                "Organizational culture", "Decision-making processes"
            ],
            "keywords": ["revenue", "profit", "market", "strategy", "management"],
            "question_types": ["case study", "business scenario"]
        },
        "philosophy": {
            "topics": [
                "Epistemology", "Metaphysics", "Ethics", "Logic", "Aesthetics",
                "Philosophy of mind", "Philosophy of language", "Political philosophy",
                "Philosophy of science", "Ontology", "Phenomenology", "Existentialism",
                "Ancient philosophy", "Modern philosophy", "Virtue ethics"
            ],
            "keywords": ["knowledge", "being", "reality", "consciousness", "truth"],
            "question_types": ["philosophical argument analysis", "concept comparison"]
        },
        "history": {
            "topics": [
                "Ancient history", "Medieval history", "Modern history", "World War I",
                "World War II", "Cold War", "Industrial Revolution", "American history",
                "European history", "Asian history", "African history",
                "Cultural history", "Intellectual history", "Economic history"
            ],
            "keywords": ["war", "revolution", "empire", "civilization", "century"],
            "question_types": ["chronological ordering", "cause-effect analysis"]
        }
    }

    return category_insights

category_insights = analyze_mmlu_pro_categories()

In [ ]:
# @title 1-2: Wikipedia API를 이용한 문서 수집

def collect_wikipedia_documents(category, topics, language='en'):
    """Wikipedia API를 사용하여 문서 수집"""
    wiki = wikipediaapi.Wikipedia(
        user_agent='RAG_Project/1.0 (ewha_AI_NLP_7)',
        language=language
    )

    documents = []

    for topic in topics:
        print(f"  → {category}: {topic}...", end=" ")
        try:
            page = wiki.page(topic)

            if not page.exists():
                print("❌")
                continue

            if page.fullurl.endswith("_(disambiguation)"):
                print("⊘")
                continue

            doc = Document(
                page_content=page.text,
                metadata={
                    "source": f"Wikipedia: {topic}",
                    "url": page.fullurl,
                    "category": category,
                    "type": "wikipedia"
                }
            )
            documents.append(doc)
            print(f"✓")

        except Exception as e:
            print(f"✗")
            continue

        time.sleep(1)

    return documents

# Wikipedia 문서 수집
wikipedia_collection = {}

for category, insights in category_insights.items():
    print(f"\n【 {category.upper()} 】")
    docs = collect_wikipedia_documents(category, insights["topics"][:], language='en')
    wikipedia_collection[category] = docs
    print(f"  합계: {len(docs)}개")

all_wikipedia_docs = []
for docs in wikipedia_collection.values():
    all_wikipedia_docs.extend(docs)

print(f"\n총 Wikipedia 문서 수집: {len(all_wikipedia_docs)}개")

In [ ]:
# @title 1-3: Stanford Encyclopedia of Philosophy 크롤링

def scrape_stanford_sep_entry_structured(url, title):
    """Stanford SEP 항목을 섹션별로 크롤링"""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'

        if response.status_code != 200:
            return []

        soup = BeautifulSoup(response.content, 'html.parser')
        content_div = soup.find('div', id='main-text')

        if not content_div:
            return []

        docs = []

        # 각 h2, h3 헤딩을 섹션으로 분리
        current_section = None
        current_section_content = []
        current_subsection = None

        for element in content_div.find_all(['h2', 'h3', 'p', 'ul', 'ol']):
            # h2는 메인 섹션
            if element.name == 'h2':
                # 이전 섹션 저장
                if current_section and current_section_content:
                    section_text = '\n'.join(current_section_content)
                    if len(section_text) > 100:
                        doc = Document(
                            page_content=section_text,
                            metadata={
                                "source": f"Stanford SEP: {title}",
                                "url": url,
                                "title": title,
                                "section": current_section,
                                "subsection": current_subsection,
                                "category": "philosophy",
                                "type": "stanford_sep"
                            }
                        )
                        docs.append(doc)

                current_section = element.get_text(strip=True)
                current_subsection = None
                current_section_content = []

            # h3는 서브섹션
            elif element.name == 'h3':
                # 이전 서브섹션 저장
                if current_section and current_section_content:
                    section_text = '\n'.join(current_section_content)
                    if len(section_text) > 100:
                        doc = Document(
                            page_content=section_text,
                            metadata={
                                "source": f"Stanford SEP: {title}",
                                "url": url,
                                "title": title,
                                "section": current_section,
                                "subsection": current_subsection,
                                "category": "philosophy",
                                "type": "stanford_sep"
                            }
                        )
                        docs.append(doc)

                current_subsection = element.get_text(strip=True)
                current_section_content = []

            # 본문 내용
            elif element.name in ['p', 'ul', 'ol']:
                text = element.get_text(strip=True)
                if text:
                    current_section_content.append(text)

        # 마지막 섹션 저장
        if current_section and current_section_content:
            section_text = '\n'.join(current_section_content)
            if len(section_text) > 100:
                doc = Document(
                    page_content=section_text,
                    metadata={
                        "source": f"Stanford SEP: {title}",
                        "url": url,
                        "title": title,
                        "section": current_section,
                        "subsection": current_subsection,
                        "category": "philosophy",
                        "type": "stanford_sep"
                    }
                )
                docs.append(doc)

        return docs

    except Exception as e:
        print(f"오류: {str(e)}")
        return []


# Stanford SEP 항목
stanford_sep_urls = {
    "Epistemology": "https://plato.stanford.edu/entries/epistemology/",
    "Metaphysics": "https://plato.stanford.edu/entries/metaphysics/",

    "Virtue Ethics": "https://plato.stanford.edu/entries/ethics-virtue/",
    "Deontological Ethics": "https://plato.stanford.edu/entries/ethics-deontological/",

    "Classical Logic": "https://plato.stanford.edu/entries/logic-classical/",
    "Propositional Logic": "https://plato.stanford.edu/entries/logic-propositional/",

    "Beauty": "https://plato.stanford.edu/entries/beauty/",
    "The Concept of the Aesthetic": "https://plato.stanford.edu/entries/aesthetic-concept/",

    "Consciousness and Intentionality": "https://plato.stanford.edu/entries/consciousness-intentionality/",

    "Justice": "https://plato.stanford.edu/entries/justice/",

    "Ontological Dependence": "https://plato.stanford.edu/entries/dependence-ontological/",

    "Theories of Meaning": "https://plato.stanford.edu/entries/meaning/",

    "The History of Utilitarianism": "https://plato.stanford.edu/entries/utilitarianism-history/",
    "Mill’s Moral and Political Philosophy": "https://plato.stanford.edu/entries/mill-moral-political/",
    "Consequentialism": "https://plato.stanford.edu/entries/consequentialism/",
    "Rule Consequentialism": "https://plato.stanford.edu/entries/consequentialism-rule/",
    "Jeremy Bentham": "https://plato.stanford.edu/entries/bentham/",

    "Free Will": "https://plato.stanford.edu/entries/freewill/",
    "Possible Worlds": "https://plato.stanford.edu/entries/possible-worlds/",
    "Rights": "https://plato.stanford.edu/entries/rights/",

    "Original Position": "https://plato.stanford.edu/entries/original-position/",
    "Contractarianism": "https://plato.stanford.edu/entries/contractarianism/",
    "Contemporary Approaches to the Social Contract": "https://plato.stanford.edu/entries/contractarianism-contemporary/",
    "Hobbes’s Moral and Political Philosophy": "https://plato.stanford.edu/entries/hobbes-moral/",
    "Contractualism": "https://plato.stanford.edu/entries/contractualism/",

    "Artificial Intelligence": "https://plato.stanford.edu/entries/artificial-intelligence/",
    "Personal Identity": "https://plato.stanford.edu/entries/identity-personal/",
    "Moral Relativism": "https://plato.stanford.edu/entries/moral-relativism/",
    "Skepticism": "https://plato.stanford.edu/entries/skepticism/",
    "Logical Consequence": "https://plato.stanford.edu/entries/logical-consequence/"
}

print("\n【 Stanford Encyclopedia of Philosophy 】")
stanford_sep_docs = []

for title, url in stanford_sep_urls.items():
    print(f"  → {title}...", end=" ")

    # 섹션 단위로 크롤링 (청킹 제거!)
    section_docs = scrape_stanford_sep_entry_structured(url, title)

    if section_docs:
        stanford_sep_docs.extend(section_docs)
        print(f"✓ ({len(section_docs)}개 섹션)")
    else:
        print(f"✗")

    time.sleep(2)

print(f"\n총 Stanford SEP 문서 수집: {len(stanford_sep_docs)}개")

In [ ]:
# @title 1-4: 하이브리드 청킹 구현: 문서 특성별 최적화

def apply_document_specific_chunking(documents):
    """
    문서 타입별로 최적화된 청킹 전략 적용

    - EWHA (학칙): 작은 청크 - 조문 단위로 정확히 분리
    - Wikipedia: 중간 청크 - 섹션 독립성 유지
    - Stanford SEP: 섹션 유지 + 필요시만 청킹
    """

    all_chunks = []

    for doc in documents:
        doc_type = doc.metadata.get('type', 'unknown')

        # ===================================================================
        # Wikipedia 문서 청킹 전략
        # ===================================================================
        if doc_type == 'wikipedia':
            # Wikipedia는 백과사전: 섹션이 비교적 독립적
            # 중간 크기 청크로 섹션 내용 유지
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=1024,         # 중간 청크 (섹션 단위)
                chunk_overlap=150,       # 섹션 간 컨텍스트 보존
                separators=[
                    "\n\n",              # 단락 구분
                    "\n",                # 줄바꿈
                    ".",                 # 문장 끝
                    "!",
                    "?",
                    " ",
                    ""
                ]
            )

        # ===================================================================
        # Stanford SEP 문서 청킹 전략
        # ===================================================================
        elif doc_type == 'stanford_sep':
            # Stanford SEP는 이미 섹션 단위로 분리됨
            # 길이가 2000자 넘는 섹션만 추가 청킹
            if len(doc.page_content) > 2000:
                splitter = RecursiveCharacterTextSplitter(
                    chunk_size=1500,     # 섹션 내 청킹 (철학 논증 보존)
                    chunk_overlap=200,   # 논증 연결성 유지
                    separators=[
                        "\n\n",          # 단락 구분 (철학 논증 단위)
                        "\n",
                        ".",
                        " ",
                        ""
                    ]
                )
            else:
                # 짧은 섹션은 그대로 유지 (논리적 완결성)
                all_chunks.append(doc)
                continue

        # ===================================================================
        # 기타 문서 타입
        # ===================================================================
        else:
            # 기본 전략
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=1024,
                chunk_overlap=150,
                separators=["\n\n", "\n", ".", " ", ""]
            )

        # ===================================================================
        # 청크 분할 및 메타데이터 추가
        # ===================================================================
        chunks = splitter.split_text(doc.page_content)

        for i, chunk in enumerate(chunks):
            new_doc = Document(
                page_content=chunk,
                metadata={
                    **doc.metadata,
                    "chunk_index": i,
                    "total_chunks": len(chunks),
                    "chunking_strategy": f"{doc_type}_optimized",
                    "original_length": len(doc.page_content)
                }
            )
            all_chunks.append(new_doc)

    return all_chunks

In [ ]:
# @title 1-5: 문서 타입 메타데이터 지정

# Wikipedia 문서 타입 확인
print("\n【 Wikipedia 문서 】")
for doc in all_wikipedia_docs:
    if 'type' not in doc.metadata:
        doc.metadata['type'] = 'wikipedia'

print(f"✓ Wikipedia 문서: {len(all_wikipedia_docs)}개 (type='wikipedia' 확인)")

# Stanford SEP 문서 타입 확인
print("\n【 Stanford SEP 문서 】")
for doc in stanford_sep_docs:
    if 'type' not in doc.metadata:
        doc.metadata['type'] = 'stanford_sep'

print(f"✓ Stanford SEP 문서: {len(stanford_sep_docs)}개 (type='stanford_sep' 확인)")

In [ ]:
# @title 1-6: 학문 자료 (Wikipedia + Stanford SEP) 청킹

print("\n【 학문 자료 청킹 】")
academic_raw_docs = []
academic_raw_docs.extend(all_wikipedia_docs)
academic_raw_docs.extend(stanford_sep_docs)

academic_chunks = apply_document_specific_chunking(academic_raw_docs)
print(f"✓ 학문 자료 청크: {len(academic_chunks)}개")

# 학문 자료 청크 분포
from collections import Counter
academic_chunk_stats = Counter()
for chunk in academic_chunks:
    academic_chunk_stats[chunk.metadata.get('type', 'unknown')] += 1

print(f"\n  청크 분포:")
for doc_type, count in academic_chunk_stats.items():
    print(f"    - {doc_type}: {count}개")

# 평균 크기
wiki_chunks = [c for c in academic_chunks if c.metadata.get('type') == 'wikipedia']
sep_chunks = [c for c in academic_chunks if c.metadata.get('type') == 'stanford_sep']

if wiki_chunks:
    avg_wiki = sum(len(c.page_content) for c in wiki_chunks) / len(wiki_chunks)
    print(f"\n  Wikipedia 평균 청크 크기: {avg_wiki:.0f}자")

if sep_chunks:
    avg_sep = sum(len(c.page_content) for c in sep_chunks) / len(sep_chunks)
    print(f"  Stanford SEP 평균 청크 크기: {avg_sep:.0f}자")

In [ ]:
# @title 1-7: 임베딩 생성

upstage_embeddings = UpstageEmbeddings(
    api_key=UPSTAGE_API_KEY,
    model="solar-embedding-1-large"
)

# ===================================================================
# 학문 자료 벡터스토어 (MMLU-Pro 전용)
# ===================================================================
print("\n【 학문 자료 벡터스토어 생성 】")
print(f"  청크 수: {len(academic_chunks)}개")
print(f"  포함: Wikipedia ({len(wiki_chunks)}개) + Stanford SEP ({len(sep_chunks)}개)")

# 학문 자료 Dense 임베딩 (FAISS)
academic_vectorstore = FAISS.from_documents(academic_chunks, upstage_embeddings)
print("✓ 학문 자료 Dense 임베딩 (FAISS) 생성 완료")

# 학문 자료 Sparse 검색 (BM25)
academic_sparse_retriever = BM25Retriever.from_documents(academic_chunks)
academic_sparse_retriever.k = 5
print("✓ 학문 자료 Sparse (BM25) 생성 완료")

In [ ]:
# @title 1-8: 임베딩 파일 저장

# ===================================================================
# 학문 자료 벡터스토어 저장
# ===================================================================
print("\n【 학문 자료 벡터스토어 저장 】")

# FAISS 임베딩 저장
academic_embedding_path = os.path.join(embedding_save_path, "academic-vectorstore")
academic_vectorstore.save_local(academic_embedding_path)
print(f"✓ 학문 자료 FAISS 저장: {academic_embedding_path}")

# BM25 Retriever 저장
academic_bm25_path = os.path.join(embedding_save_path, "academic_bm25_retriever.pkl")
with open(academic_bm25_path, 'wb') as f:
    pickle.dump(academic_sparse_retriever, f)
print(f"✓ 학문 자료 BM25 저장: {academic_bm25_path}")

# ===================================================================
# 메타데이터 저장
# ===================================================================
print("\n【 메타데이터 저장 】")

metadata = {
    "strategy": "separated_vectorstores",
    "description": "EWHA와 학문 자료를 분리하여 문제 유형별 RAG 최적화",

    "academic": {
        "purpose": "영어 MMLU-Pro 문제 전용 (law, psychology, business, philosophy, history)",
        "num_chunks": len(academic_chunks),
        "vectorstore_path": academic_embedding_path,
        "bm25_path": academic_bm25_path,
        "embedding_model": "solar-embedding-1-large",
        "chunking": {
            "wikipedia": {
                "strategy": "wikipedia_optimized",
                "chunk_size": 1024,
                "chunk_overlap": 150,
                "chunks": len(wiki_chunks),
                "avg_chunk_size": int(avg_wiki) if wiki_chunks else 0
            },
            "stanford_sep": {
                "strategy": "stanford_sep_adaptive",
                "chunk_size": "1500 or original",
                "chunk_overlap": 200,
                "chunks": len(sep_chunks),
                "avg_chunk_size": int(avg_sep) if sep_chunks else 0
            }
        },
        "source_documents": {
            "wikipedia": len(all_wikipedia_docs),
            "stanford_sep": len(stanford_sep_docs)
        }
    },

    "usage": {
        "ewha_retriever": "한국어 문제 감지 시 사용",
        "academic_retriever": "영어 MMLU-Pro 문제 감지 시 사용",
        "hybrid_search": "각 벡터스토어에서 Dense(FAISS) + Sparse(BM25) 앙상블"
    }
}

metadata_path = os.path.join(embedding_save_path, "metadata.json")
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"✓ 메타데이터 저장: {metadata_path}")

print("\n" + "=" * 70)
print("✓ 분리된 벡터스토어 생성 및 저장 완료!")
print("=" * 70)

print("\n【 저장된 파일 요약 】")
print(f"\n1. 학문 자료 벡터스토어:")
print(f"   - FAISS: {academic_embedding_path}")
print(f"   - BM25: {academic_bm25_path}")
print(f"   - 청크 수: {len(academic_chunks)}개")
print(f"\n2. 메타데이터: {metadata_path}")


# ===================================================================
# 검증: 샘플 청크 확인
# ===================================================================
print("\n" + "=" * 70)
print("【 샘플 청크 검증 】")
print("=" * 70)

print("\n[학문 자료 청크 샘플]")
for i, chunk in enumerate(academic_chunks[:2], 1):
    print(f"\n청크 {i}:")
    print(f"  타입: {chunk.metadata.get('type')}")
    print(f"  전략: {chunk.metadata.get('chunking_strategy')}")
    print(f"  출처: {chunk.metadata.get('source')}")
    print(f"  카테고리: {chunk.metadata.get('category', 'N/A')}")
    print(f"  섹션: {chunk.metadata.get('section', 'N/A')}")
    print(f"  길이: {len(chunk.page_content)}자")
    print(f"  내용: {chunk.page_content[:120]}...")

# 2. Parsing+Embedding: ewha.pdf



⭐ 문서가 이미 어느 정도 구조화 및 분할되어 있다고 판단하면 `RecursiveCharacterTextSplitter` 같은 단순 문자 기반 분할기를 사용할 때 chunk_size, chunk_overlap 같은 파라미터를 크게 조정하여 중복 최소, 단위 그대로를 유지하도록 활용합니다.

반면, 문서 내 구조가 불분명하거나 하나의 덩어리가 너무 크다면, 더 정교한 구조 기반 분할(예: 제목 단위, 문단 단위 분할)이나 `Upstage Document Parse` 같이 레이아웃과 의미를 파악한 후 분할하는 방식을 권장합니다.

In [ ]:
# @title 2-1: EWHA.pdf를 Document Parse API로 로드

def parse_ewha_with_upstage_api(pdf_path, api_key):
    """
    Upstage Document Parse API로 ewha.pdf 파싱
    → 테이블 구조까지 정확히 인식
    """
    url = "https://api.upstage.ai/v1/document-ai/document-parse"

    headers = {"Authorization": f"Bearer {api_key}"}

    data = {
        "ocr": "force",              # OCR 강제 적용 (테이블 정확도 향상)
        "output_formats": "markdown" # Markdown 형식 (LangChain 호환)
    }

    print(f"Document Parse API 호출 중...")
    print(f"  파일: {pdf_path}")
    print(f"  설정: ocr=force, output_formats=markdown")

    try:
        with open(pdf_path, "rb") as f:
            files = {"document": f}
            response = requests.post(url, headers=headers, files=files, data=data)

        if response.status_code != 200:
            print(f"❌ API 에러: {response.status_code}")
            print(f"응답: {response.text}")
            return None

        result = response.json()

        # Markdown 콘텐츠 추출
        markdown_content = result.get("content", {}).get("markdown", "")

        if not markdown_content:
            print("❌ Markdown 콘텐츠가 비어있습니다.")
            return None

        # 파싱 결과 저장 (디버깅용)
        parsed_file_path = os.path.join(data_path, "data/raw/ewha_parsed.md")
        with open(parsed_file_path, "w", encoding="utf-8") as f:
            f.write(markdown_content)

        print(f"✓ Document Parse 완료!")
        print(f"  총 {len(markdown_content):,}자 추출")
        print(f"  저장 경로: {parsed_file_path}")

        # 테이블 인식 확인
        html_content = result.get("content", {}).get("html", "")
        if "<table>" in html_content or "|" in markdown_content:
            print(f"✓ 테이블 구조 정상 인식됨")

        return markdown_content

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {pdf_path}")
        return None
    except Exception as e:
        print(f"❌ 예외 발생: {str(e)}")
        return None


# Document Parse API로 ewha.pdf 파싱
ewha_pdf_path = os.path.join(data_path, 'data/raw/ewha.pdf')
ewha_markdown = parse_ewha_with_upstage_api(ewha_pdf_path, UPSTAGE_API_KEY)

if ewha_markdown is None:
    print("\n⚠️  Document Parse 실패! PyPDFLoader로 폴백합니다...")
    from langchain_community.document_loaders import PyPDFLoader
    ewha_loader = PyPDFLoader(ewha_pdf_path)
    ewha_docs = ewha_loader.load()
    print(f"✓ PyPDFLoader로 로드 완료: {len(ewha_docs)}개 페이지")
else:
    # Markdown을 LangChain Document로 변환
    ewha_docs = [
        Document(
            page_content=ewha_markdown,
            metadata={
                "source": "EWHA University Regulations",
                "type": "ewha",
                "parsing_method": "upstage_document_parse",
                "format": "markdown"
            }
        )
    ]
    print(f"✓ LangChain Document 생성 완료: 1개 문서")

print(f"✓ EWHA 문서 로드 완료!")

In [ ]:
# @title 2-2: 하이브리드 청킹 구현: 문서 특성별 최적화

def apply_document_specific_chunking(documents):
    """
    문서 타입별로 최적화된 청킹 전략 적용

    - EWHA (학칙): 작은 청크 - 조문 단위로 정확히 분리
    - Wikipedia: 중간 청크 - 섹션 독립성 유지
    - Stanford SEP: 섹션 유지 + 필요시만 청킹
    """

    all_chunks = []

    for doc in documents:
        doc_type = doc.metadata.get('type', 'unknown')

        # ===================================================================
        # EWHA 문서 청킹 전략
        # ===================================================================
        if doc_type == 'ewha':
            # EWHA는 학칙 문서: 조문(제XX조) 단위로 분리 필요
            # 작은 청크로 "제5조", "입학정원" 등 정확한 검색 가능
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=512,          # 작은 청크 (조문 1-2개)
                chunk_overlap=100,       # 조문 간 연결성 유지
                separators=[
                    "\n제",               # "제XX조" 구분자 최우선
                    "\n\n",              # 단락 구분
                    "\n",                # 줄바꿈
                    ".",                 # 문장 끝
                    " ",                 # 단어
                    ""
                ]
            )

        # ===================================================================
        # 기타 문서 타입
        # ===================================================================
        else:
            # 기본 전략
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=1024,
                chunk_overlap=150,
                separators=["\n\n", "\n", ".", " ", ""]
            )

        # ===================================================================
        # 청크 분할 및 메타데이터 추가
        # ===================================================================
        chunks = splitter.split_text(doc.page_content)

        for i, chunk in enumerate(chunks):
            new_doc = Document(
                page_content=chunk,
                metadata={
                    **doc.metadata,
                    "chunk_index": i,
                    "total_chunks": len(chunks),
                    "chunking_strategy": f"{doc_type}_optimized",
                    "original_length": len(doc.page_content)
                }
            )
            all_chunks.append(new_doc)

    return all_chunks

In [ ]:
# @title 2-3: EWHA 문서: 파싱된 Markdown 파일 로드

print("【 EWHA 문서 】")

# Document Parse API로 생성된 Markdown 파일 경로
parsed_md_path = os.path.join(data_path, "data/raw/ewha_parsed.md")

# Markdown 파일이 존재하는지 확인
if os.path.exists(parsed_md_path):
    print(f"✓ 파싱된 Markdown 파일 발견: {parsed_md_path}")

    # Markdown 파일 읽기
    with open(parsed_md_path, 'r', encoding='utf-8') as f:
        ewha_markdown_content = f.read()

    print(f"  파일 크기: {len(ewha_markdown_content):,}자")

    # LangChain Document 객체 생성
    ewha_docs = [
        Document(
            page_content=ewha_markdown_content,
            metadata={
                "source": "EWHA University Regulations",
                "type": "ewha",
                "parsing_method": "upstage_document_parse",
                "format": "markdown",
                "parsed_file": "ewha_parsed2.md"
            }
        )
    ]

    print(f"✓ EWHA 문서 생성 완료: {len(ewha_docs)}개 (type='ewha' 지정)")
    print(f"  파싱 방법: upstage_document_parse")
    print(f"  형식: markdown")

    # 테이블 및 조문 통계
    table_count = ewha_markdown_content.count("|")
    article_count = len(re.findall(r'제\s*\d+\s*조', ewha_markdown_content))

    print(f"  테이블 구분자 '|': {table_count}개")
    print(f"  조문 '제XX조': {article_count}개")

    # 샘플 출력
    print(f"\n  처음 200자:")
    print(f"  {ewha_markdown_content[:200]}...")

else:
    # Markdown 파일이 없으면 PART 1-2에서 생성된 ewha_docs 사용
    print(f"⚠️  파싱 파일을 찾을 수 없습니다: {parsed_md_path}")

    if 'ewha_docs' in globals() and ewha_docs:
        print(f"✓ 메모리에서 ewha_docs 사용: {len(ewha_docs)}개")

        # 메타데이터 확인 및 설정
        for doc in ewha_docs:
            if 'type' not in doc.metadata:
                doc.metadata['type'] = 'ewha'
            if 'source' not in doc.metadata:
                doc.metadata['source'] = 'EWHA University Regulations'

        print(f"  타입: {ewha_docs[0].metadata.get('type')}")
        print(f"  파싱 방법: {ewha_docs[0].metadata.get('parsing_method', 'N/A')}")
    else:
        # 최후의 수단: PyPDFLoader 폴백
        print("⚠️  PyPDFLoader로 폴백합니다...")
        from langchain_community.document_loaders import PyPDFLoader
        ewha_pdf_path = os.path.join(data_path, 'ewha.pdf')
        ewha_loader = PyPDFLoader(ewha_pdf_path)
        ewha_docs = ewha_loader.load()

        for doc in ewha_docs:
            doc.metadata['type'] = 'ewha'
            doc.metadata['source'] = 'EWHA University Regulations'

        print(f"✓ PyPDFLoader로 로드 완료: {len(ewha_docs)}개 페이지")

In [ ]:
# @title 2-4: EWHA 문서만 따로 청킹 (Document Parse 결과 청킹)

print("\n【 EWHA 문서 청킹 】")
print(f"  입력: {len(ewha_docs)}개 문서")
print(f"  전략: ewha_optimized (제XX조 단위 분리)")

ewha_chunks = apply_document_specific_chunking(ewha_docs)
print(f"✓ EWHA 청크: {len(ewha_chunks)}개")

if ewha_chunks:
    avg_ewha = sum(len(c.page_content) for c in ewha_chunks) / len(ewha_chunks)
    print(f"  평균 청크 크기: {avg_ewha:.0f}자")

    # 청킹 품질 확인
    article_chunks = [c for c in ewha_chunks if '제' in c.page_content[:10]]
    print(f"  '제XX조' 시작 청크: {len(article_chunks)}개")

In [ ]:
# @title 2-5: 임베딩 생성

upstage_embeddings = UpstageEmbeddings(
    api_key=UPSTAGE_API_KEY,
    model="solar-embedding-1-large"
)

# ===================================================================
# EWHA 벡터스토어 (한국어 학칙 전용)
# ===================================================================
print("\n【 EWHA 벡터스토어 생성 】")
print(f"  청크 수: {len(ewha_chunks)}개")
print(f"  출처: Document Parse API (테이블 구조 보존)")

# EWHA Dense 임베딩 (FAISS)
ewha_vectorstore = FAISS.from_documents(ewha_chunks, upstage_embeddings)
print("✓ EWHA Dense 임베딩 (FAISS) 생성 완료")

# EWHA Sparse 검색 (BM25)
ewha_sparse_retriever = BM25Retriever.from_documents(ewha_chunks)
ewha_sparse_retriever.k = 5
print("✓ EWHA Sparse (BM25) 생성 완료")

In [ ]:
# @title 2-6: 임베딩 파일 저장

# ===================================================================
# EWHA 벡터스토어 저장
# ===================================================================
print("\n【 EWHA 벡터스토어 저장 】")

# FAISS 임베딩 저장
ewha_embedding_path = os.path.join(embedding_save_path, "ewha-vectorstore")
ewha_vectorstore.save_local(ewha_embedding_path)
print(f"✓ EWHA FAISS 저장: {ewha_embedding_path}")

# BM25 Retriever 저장
ewha_bm25_path = os.path.join(embedding_save_path, "ewha_bm25_retriever.pkl")
with open(ewha_bm25_path, 'wb') as f:
    pickle.dump(ewha_sparse_retriever, f)
print(f"✓ EWHA BM25 저장: {ewha_bm25_path}")

# ===================================================================
# 메타데이터 저장
# ===================================================================
print("\n【 메타데이터 저장 】")

metadata = {
    "strategy": "separated_vectorstores",
    "description": "EWHA와 학문 자료를 분리하여 문제 유형별 RAG 최적화",

    "ewha": {
        "purpose": "한국어 이화여대 학칙 문제 전용",
        "num_chunks": len(ewha_chunks),
        "vectorstore_path": ewha_embedding_path,
        "bm25_path": ewha_bm25_path,
        "embedding_model": "solar-embedding-1-large",
        "chunking": {
            "strategy": "ewha_optimized",
            "chunk_size": 512,
            "chunk_overlap": 100,
            "separators": ["\\n제", "\\n\\n", "\\n", ".", " "],
            "avg_chunk_size": int(avg_ewha) if ewha_chunks else 0
        },
        "source_documents": len(ewha_docs)
    },

    "usage": {
        "ewha_retriever": "한국어 문제 감지 시 사용",
        "academic_retriever": "영어 MMLU-Pro 문제 감지 시 사용",
        "hybrid_search": "각 벡터스토어에서 Dense(FAISS) + Sparse(BM25) 앙상블"
    }
}

metadata_path = os.path.join(embedding_save_path, "metadata.json")
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"✓ 메타데이터 저장: {metadata_path}")

print("\n" + "=" * 70)
print("✓ 분리된 벡터스토어 생성 및 저장 완료!")
print("=" * 70)

print("\n【 저장된 파일 요약 】")
print(f"\n1. EWHA 벡터스토어:")
print(f"   - FAISS: {ewha_embedding_path}")
print(f"   - BM25: {ewha_bm25_path}")
print(f"   - 청크 수: {len(ewha_chunks)}개")
print(f"\n2. 메타데이터: {metadata_path}")


# ===================================================================
# 검증: 샘플 청크 확인
# ===================================================================
print("\n" + "=" * 70)
print("【 샘플 청크 검증 】")
print("=" * 70)

print("\n[EWHA 청크 샘플]")
for i, chunk in enumerate(ewha_chunks[:2], 1):
    print(f"\n청크 {i}:")
    print(f"  타입: {chunk.metadata.get('type')}")
    print(f"  전략: {chunk.metadata.get('chunking_strategy')}")
    print(f"  출처: {chunk.metadata.get('source')}")
    print(f"  길이: {len(chunk.page_content)}자")
    print(f"  내용: {chunk.page_content[:120]}...")

## Table 추가 전처리

In [ ]:
import re
import json
from typing import List, Dict, Tuple
from langchain.schema import Document

class ImprovedTablePreprocessor:
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.table_metadata_list = []

    def extract_tables(self, md_content: str) -> List[Dict]:
        base_tables = []

        table_pattern = r'\[별표 (\d+)\](.*?)(?=\n\[별표|\Z)'
        matches = list(re.finditer(table_pattern, md_content, re.DOTALL))

        for match in matches:
            table_num = match.group(1)
            block_text = match.group(2).strip()

            # ✅ (경과조치)가 붙은 [별표 2]는 전부 스킵
            if table_num == '2' and '(경과조치' in block_text:
                if self.verbose:
                    print("   → [별표 2] (경과조치) 블록은 테이블 추출에서 제외")
                continue

            # [별표 1]이면 다년차 전용 파서
            if table_num == '1':
                multi = self._extract_multi_tables(block_text)
                base_tables.extend(multi)
                continue

            # 나머지 [별표]는 기존 방식
            lines = block_text.split('\n')

            year_info = ""
            for line in lines[:5]:
                year_match = re.search(r'(\d{4})학년도', line)
                if year_match:
                    year_info = year_match.group(1)
                    break

            table_start = None
            for i, line in enumerate(lines):
                if line.strip().startswith('|') and line.strip().endswith('|'):
                    table_start = i
                    break
            if table_start is None:
                continue

            header_line = lines[table_start]
            headers = [h.strip() for h in header_line.split('|')[1:-1]]

            data_rows = []
            for i in range(table_start + 2, len(lines)):
                line = lines[i].strip()
                if not line.startswith('|'):
                    break
                cells = [cell.strip() for cell in line.split('|')[1:-1]]
                if len(cells) < len(headers):
                    cells += [""] * (len(headers) - len(cells))
                data_rows.append(cells)

            if data_rows:
                base_tables.append({
                    'num': table_num,
                    'headers': headers,
                    'rows': data_rows,
                    'year': int(year_info) if year_info else None
                })

            if self.verbose:
                    print(f"   ✓ [별표 {table_num}] {year_info}학년도 테이블: "
                            f"{len(data_rows)}행 x {len(headers)}열")

        if self.verbose:
            print(f"   총 {len(base_tables)}개 테이블 추출")

        return base_tables

    def _extract_multi_tables(self, block_text: str, verbose=True) -> List[Dict]:
        """
        이미 [별표 1] 내부 텍스트만 들어온 상태에서,
        '입학정원(XXXX학년도)' 제목 줄 + 바로 뒤 |...| 표를 연도별 서브테이블로 쪼갠다.
        """

        lines = block_text.split('\n')
        all_tables = []
        base_num = '1'  # [별표 1] 전용

        current_year = None
        i = 0
        while i < len(lines):
            line = lines[i].strip()

            # 1) '입학정원(2019학년도)' 같은 제목 줄에서 연도 캡쳐
            year_match = re.search(r'입학정원\((\d{4})학년도\)', line)
            if year_match:
                current_year = int(year_match.group(1))
                i += 1
                continue

            # 2) 헤더 줄(| ... |) 감지
            if line.startswith('|') and line.endswith('|'):
                if current_year is None:
                    i += 1
                    continue

                header_line = line
                headers = [h.strip() for h in header_line.split('|')[1:-1]]
                if len(headers) < 2:
                    i += 1
                    continue

                # 3) 이어지는 데이터 행 수집
                data_rows = []
                j = i + 2  # 헤더 아래 구분선 한 줄 건너뛰기 가정
                while j < len(lines):
                    row_line = lines[j].strip()
                    if not (row_line.startswith('|') and row_line.endswith('|')):
                        break
                    cells = [c.strip() for c in row_line.split('|')[1:-1]]
                    if len(cells) < len(headers):
                        cells += [""] * (len(headers) - len(cells))
                    data_rows.append(cells)
                    j += 1

                if data_rows:
                    table_id = f"{base_num}-{current_year}"
                    all_tables.append({
                        'num': table_id,
                        'base_num': base_num,
                        'headers': headers,
                        'rows': data_rows,
                        'year': current_year,
                    })
                    if verbose:
                        print(f"   ✓ [별표 {base_num}] {current_year}학년도 테이블: "
                            f"{len(data_rows)}행 x {len(headers)}열")

                i = j
                continue

            i += 1

        return all_tables

    def normalize_table_to_metadata(self, table: Dict) -> List[Dict]:
        """
        📌 핵심: 동적 컬럼 매핑 (정원, 학위, 학과명 자동 감지)
        """
        headers = table['headers']
        rows = table['rows']
        year = table['year']
        table_id = table['num']

        # 1. 컬럼 인덱스 자동 찾기
        cols = {'college': -1, 'dept': -1, 'quota': -1, 'degree': -1}

        for i, h in enumerate(headers):
            h_clean = h.replace(" ", "")
            if '대학' in h_clean and '학위' not in h_clean: cols['college'] = i
            elif '학부' in h_clean or '학과' in h_clean or '전공' in h_clean: cols['dept'] = i
            elif '정원' in h_clean: cols['quota'] = i
            elif '학위' in h_clean or '종류' in h_clean: cols['degree'] = i # <--- 학위 컬럼 감지

        metadata_list = []

        for row_idx, row in enumerate(rows):
            try:
                # 데이터 추출
                college = row[cols['college']].strip() if cols['college'] != -1 else ""
                dept_text = row[cols['dept']].strip() if cols['dept'] != -1 else ""

                # 정원 추출 (없으면 0)
                quota = 0
                if cols['quota'] != -1:
                    quota_str = row[cols['quota']].strip()
                    quota_match = re.search(r'\d+', quota_str.replace(',', ''))
                    if quota_match: quota = int(quota_match.group())

                # 학위 추출 (없으면 빈 문자열)
                degree = ""
                if cols['degree'] != -1:
                    degree = row[cols['degree']].strip()

                # "소계" 처리
                if '소계' in dept_text or '합계' in dept_text:
                    metadata_list.append({
                        'type': 'subtotal',
                        'year': year,
                        'college': college,
                        'label': dept_text,
                        'quota': quota,
                        'degree': degree,
                        'table_id': table_id
                    })
                    continue

                # 학과 여러 개 뭉친 경우 처리 (단, 학위 정보가 있으면 쪼개면 안됨 -> 학위는 보통 1:1)
                # 여기서는 안전하게 "단일 학과"로 가정하되, 쉼표로 나열된 경우만 분리 고려
                # 하지만 학칙 표는 보통 줄바꿈이나 셀 병합 이슈가 있으므로, 일단 단순화해서 저장

                metadata_list.append({
                    'type': 'general_info', # 통합된 타입
                    'year': year,
                    'college': college,
                    'department': dept_text,
                    'quota': quota,
                    'degree': degree, # 학위 정보 저장
                    'table_id': table_id,
                    'row_index': row_idx
                })

            except Exception as e:
                continue

        return metadata_list

    def preprocess(self, md_content: str) -> Tuple[str, List[Dict]]:
        tables = self.extract_tables(md_content)
        all_metadata = []
        for table in tables:
            metadata = self.normalize_table_to_metadata(table)
            all_metadata.extend(metadata)

        # 원본에서 테이블 제거 (청크 중복 방지)
        processed_content = md_content
        processed_content = re.sub(r'\[별표 (\d+)\](.*?)(?=\n\[별표|\Z)', '', processed_content, flags=re.DOTALL)

        return processed_content, all_metadata

In [ ]:
def split_chunks_with_metadata(processed_content: str, table_metadata: List[Dict], verbose=True) -> List[Document]:

    # 1. 조문 청크
    regulation_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200, chunk_overlap=150,
        separators=["\n제[0-9]+조", "\n\n", "\n", ".", " "]
    )
    regulation_chunks = regulation_splitter.split_text(processed_content)
    regulation_docs = [
        Document(page_content=c, metadata={'source_type': 'regulation', 'chunk_index': i})
        for i, c in enumerate(regulation_chunks)
    ]

    # 2. 테이블 청크 (동적 문장 생성)
    tables_grouped = {}
    for meta in table_metadata:
        t_id = meta.get('table_id', meta.get('num', 'unknown'))
        if t_id not in tables_grouped: tables_grouped[t_id] = []
        tables_grouped[t_id].append(meta)

    table_docs = []
    MAX_ROWS = 40

    for t_id, rows in tables_grouped.items():
        if not rows: continue
        year = rows[0].get('year', '')

        for i in range(0, len(rows), MAX_ROWS):
            chunk_rows = rows[i : i + MAX_ROWS]

            # 청크 헤더
            chunk_text = f"## [표 {t_id} 정보 (연도: {year}, Part {i//MAX_ROWS + 1})]\n\n"

            colleges = set()
            for row in chunk_rows:
                college = row.get('college', '')
                dept = row.get('department', row.get('label', ''))
                quota = row.get('quota', 0)
                degree = row.get('degree', '')

                if college: colleges.add(college)

                # 정보 조합 (정원 + 학위)
                info_str = []
                if quota > 0: info_str.append(f"입학정원 {quota}명")
                if degree: info_str.append(f"수여학위 '{degree}'")

                # 문장 완성
                if row['type'] == 'subtotal':
                    chunk_text += f"- **{college} {dept}: {', '.join(info_str)}**\n"
                else:
                    if info_str:
                        chunk_text += f"- {college} {dept}: {', '.join(info_str)}\n"
                    else:
                        # 정보가 없어도 학과명은 기록 (존재 여부 확인용)
                        chunk_text += f"- {college} {dept} (세부 정보 없음)\n"

            table_docs.append(Document(
                page_content=chunk_text,
                metadata={
                    'source_type': 'table_full',
                    'year': year,
                    'table_id': t_id,
                    'colleges': list(colleges)
                }
            ))

    return regulation_docs + table_docs

In [ ]:
def process_and_save_all():
    print("🚀 프로세스 시작...")

    with open(os.path.join(data_path, "ewha_parsed.md"), 'r', encoding='utf-8') as f:
        md_content = f.read()

    preprocessor = ImprovedTablePreprocessor()
    processed_content, table_metadata = preprocessor.preprocess(md_content)
    all_docs = split_chunks_with_metadata(processed_content, table_metadata)

    print(f"\n💾 총 {len(all_docs)}개 청크 생성 완료")

    # 검증: 학위 정보가 잘 들어갔는지 확인
    print("\n🧐 검증: '문학사' 검색")
    for doc in all_docs:
        if "문학사" in doc.page_content and doc.metadata['source_type'] == 'table_full':
            print(f"\n[청크 샘플]\n{doc.page_content[:200]}...")
            break

    # 저장
    embeddings = UpstageEmbeddings(api_key=UPSTAGE_API_KEY, model="solar-embedding-1-large")

    print("\n⚡ FAISS 저장 중...")
    vectorstore = FAISS.from_documents(all_docs, embeddings)
    vectorstore.save_local(embedding_save_path)

    print("\n🔍 BM25 저장 중...")
    bm25 = BM25Retriever.from_documents(all_docs)
    with open(os.path.join(embedding_save_path, "ewha_bm25_retriever.pkl"), "wb") as f:
        pickle.dump(bm25, f)

    # 원본 문서 저장
    with open(os.path.join(embedding_save_path, "all_docs.pkl"), "wb") as f:
        pickle.dump(all_docs, f)

    print("\n🎉 완료!")

if __name__ == "__main__":
    process_and_save_all()

In [ ]:
# 1. 저장된 문서 로드
with open(os.path.join(embedding_save_path, "all_docs.pkl"), "rb") as f:
    saved_docs = pickle.load(f)

# 2. '수학교육과'가 포함된 모든 청크 출력 (진실의 순간)
print("=== '수학교육과' 검색 결과 ===")
for doc in saved_docs:
    if "수학교육과" in doc.page_content:
        source_type = doc.metadata.get('source_type')
        year = doc.metadata.get('year')  # 없으면 None일 수 있음
        print(f"\n[Source: {source_type}, Year: {year}]")
        idx = doc.page_content.find("수학교육과")
        print(doc.page_content[max(0, idx-100) : min(len(doc.page_content), idx+100)])

# 3. Parsing+Embedding: academic pdf files

## Import

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-upstage openai
!pip cache purge

In [ ]:
req_path = os.path.join(data_path, 'requirements.txt')
!pip install -r "{req_path}"

!pip install -qU python-dotenv PyPDF2 langchain langchain-community langchain-core langchain-text-splitters langchain_upstage oracledb python-dotenv

In [ ]:
import os
import json
import pickle

from openai import OpenAI

from langchain_upstage import (
    UpstageDocumentParseLoader,
    UpstageEmbeddings,
)
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

## 임베딩 생성 및 저장

In [ ]:
# @title 0: Configuration

embedding_save_path = os.path.join(data_path, "embeddings")

# 임베딩 저장 디렉토리 생성
os.makedirs(embedding_save_path, exist_ok=True)

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

In [ ]:
# @title 1: 모든 외부 문서 통합

import warnings
import logging
warnings.filterwarnings('ignore', category=UserWarning, module='PyPDF2')
logging.getLogger('PyPDF2').setLevel(logging.ERROR)

print("【 PDF 파일 로드 】")
academic_path = os.path.join(data_path, 'philosophy_docs')
academic_docs = []
academic_files = [
    # =====Philosophy=====
    "Intro-to-Phil-full-text.pdf",
    "Introduction to Philosophy_ Ethics.pdf"
    # ,

    # =====Law=====
    # "Business_Law_Essentials.pdf",
    # "Criminal_Law_(Lisa_Storm).pdf",
    # "Law_School_Essentials-Constitutional_Law.pdf",
    # "Exercises_in_Civil_Procedure.pdf",
    # "Tort_Law_Cases_and_Commentaries.pdf",

    # =====Psychology=====
    # "Psychology_2e.pdf",
    # "Ethical_Principles_of_Psychologists_and_Code_of_Conduct.pdf",

    # =====History=====
    # "Modern_World_History_New_Perspectives.pdf",
    # "World_History-Cultures+States+Societies_to_1500.pdf",
    # "The_History_of_Our_Tribe-Hominini.pdf",

    # =====Business=====
    # "Introduction_To_Business.pdf"

]

# **항상 text PDF 처리**
def upstage_text_loader(file_path, api_key):
    loader = UpstageDocumentParseLoader(
        api_key=api_key,
        file_path=file_path,
        ocr=False
    )

    return loader.load()

# PDF 파일들 로드
for file in academic_files:
    file_path = os.path.join(academic_path, file)
    if os.path.exists(file_path):
        print(f"\n{file}...", end=" ")
        try:
            docs = upstage_text_loader(file_path, UPSTAGE_API_KEY)
            academic_docs.extend(docs)
            print(f" ✓ ({len(docs)}개)")
        except Exception as e:
            print(f" ✗ 오류: {e}")
    else:
        print(f"\n{file}... ✗ 파일 없음")

print(f"\n{'='*70}")
print(f"최종 학문 문서: {len(academic_docs)}개")
print(f"{'='*70}")

In [ ]:
# @title 2: 청킹 (Chunking)

# 학문 자료 청킹
academic_text_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.HTML,
    chunk_size=1200,
    chunk_overlap=120,
)
academic_splits = academic_text_splitter.split_documents(academic_docs)
print(f"✓ 학문 청크: {len(academic_splits)}개")


In [ ]:
# @title 3: 임베딩 생성

upstage_embeddings = UpstageEmbeddings(
    api_key=UPSTAGE_API_KEY,
    model="solar-embedding-1-large"
)

# 학문 자료 벡터스토어
print("\n【 학문 자료 벡터스토어 생성 (임베딩) 】")
academic_vectorstore = FAISS.from_documents(academic_splits, upstage_embeddings)
print("✓ 학문 자료 Dense 임베딩 생성 완료")

academic_sparse_retriever = BM25Retriever.from_documents(academic_splits)
academic_sparse_retriever.k = 5
print("✓ 학문 자료 Sparse (BM25) 생성 완료")

In [ ]:
# @title 4: 임베딩 파일 저장

# 학문 자료 FAISS 임베딩 저장
academic_embedding_path = os.path.join(embedding_save_path, "academic_vectorstore")
academic_vectorstore.save_local(academic_embedding_path)
print(f"✓ 학문 자료 임베딩 저장: {academic_embedding_path}")

# BM25 Retriever 저장 (Pickle 형식)
academic_bm25_path = os.path.join(embedding_save_path, "academic_bm25_retriever.pkl")
with open(academic_bm25_path, 'wb') as f:
    pickle.dump(academic_sparse_retriever, f)
print(f"✓ 학문 자료 BM25 저장: {academic_bm25_path}")

# 메타데이터 저장
metadata = {
    "academic": {
        "num_chunks": len(academic_splits),
        "vectorstore_path": academic_embedding_path,
        "bm25_path": academic_bm25_path,
        "embedding_model": "solar-embedding-1-large",
        "chunk_size": 1200,
        "chunk_overlap": 120,
        "source_pdfs": academic_files + ["Judith Jarvis Thomson A Defense of Abortion.txt"]
    }
}

metadata_path = os.path.join(embedding_save_path, "metadata.json")
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ 메타데이터 저장: {metadata_path}")

print("=" * 70)
print("✓ 임베딩 생성 및 저장 완료!")


# Merge Embeddings

**NOTE ⭐**

1. `data_path="YOUR_PATH"`를 폴더 경로로 수정해주세요! 해당 경로가 사용되는 경우는 아래와 같습니다.

   * pip install의 requirements.txt 경로
   * `BASE_PATH` : 코드 셀 중 하나만 돌릴 경우도 있을 것 같아 각 코드 셀마다 BASE_PATH를 새로 정의하도록 남겨두었어요! 필요한 셀 수정해서 사용해주시면 됩니다!

---



**🔗 Vector Store 병합 (Merge)**

* FAISS dense embedding을 병합하는 코드입니다
* 폴더 내에 파일명은 `index.faiss`, `index.pkl`로 통일되어 있어야 합니다
* 출력에 나타나는 `(Added 0 chunks)` 문구는 신경쓰지 않으셔도 됩니다...! `Total Chunks`만 확인해주세용

* 가져오는 경로 수정을 원할 경우, `index.faiss`와 `index.pkl`이 들어있는 **폴더 경로**로 수정해주세요!


---



**🔠 BM25 Retriever 병합 (Combine & Re-build)**

* BM25 sparse embedding 병합하는 코드입니다

* 가져오는 경로 수정을 원할 경우, FAISS와 다르게 bm25 파일을 경로로 설정해주셔야 합니다!
---

**📝 Final Metadata JSON 생성**

* embedding 파일들의 Metadata를 저장하는 코드입니다

* **⭐ 앞에서 embedding 파일들의 "저장 경로"를 수정하셨을 경우, 반드시 수정해주셔야 하는 코드입니다! ⭐**
  
  **RAG 실행 시 metadata 파일 정보를 불러와서 embedding을 load 하기 때문에, 이후에 embedding 파일 경로 수정 시 metadata 정보도 업데이트 해주셔야 합니다!!**
  * FAISS의 저장 경로를 수정하셨을 경우, 병합 파일이 들어있는 폴더 경로로 수정해주세요
  * BM25의 저장 경로를 수정하셨을 경우, 병합 파일의 경로로 수정해주세요

* 저장 경로 수정은 자유롭게 하시면 됩니다

* 병합하여 저장 경로를 옮기지 않은 파일들은 직접 복사해서 옮겨 주셔야 합니다...

* ⭐ **metadata 파일 내에 FAISS의 chunk 개수를 살려두고 싶으신 경우** ⭐
  
  * metadata 생성 코드 실행 후, `🔎 실제 로딩된 데이터 개수 확인` 코드를 실행해주세요!
  
  * 이후, 출력 결과를 metadata 생성 코드 내 `num_chunks` 옆에 직접 작성하고 재실행 해주시면 됩니다...

## Code Cells

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-upstage openai
!pip cache purge

data_path = "YOUR_PATH"

req_path = os.path.join(data_path, 'requirements.txt')
!pip install -r "{req_path}"
!pip install bm25_retriever

### 🔗 Vector Store 병합 (Merge)


In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings
from google.colab import userdata

# 1. 설정
# API Key
os.environ["UPSTAGE_API_KEY"] = userdata.get('UPSTAGE_API_KEY')
embedding_model = UpstageEmbeddings(model="solar-embedding-1-large")

# 경로 설정
BASE_PATH = data_path
ADD_PATH = data_path

# A. 기존 벡터 스토어 경로
BASE_STORE_PATH = os.path.join(BASE_PATH, "final_embedding_4/academic-vectorstore")

# B. 추가할 벡터 스토어 경로 리스트
ADD_STORE_PATH = [
    os.path.join(ADD_PATH, "embeddings_additional/academic_vectorstore"),
]

# C. 최종 벡터 스토어 저장할 경로
FINAL_STORE_PATH = os.path.join(BASE_PATH, "final_embedding_5/academic-vectorstore")


'-', '_' 주의!

In [ ]:
# 2. 병합 로직
try:
    # 2-1. 기존 벡터 스토어 로드
    print(f"📂 Loading my store from: {BASE_STORE_PATH}")
    main_store = FAISS.load_local(
        BASE_STORE_PATH,
        embedding_model,
        allow_dangerous_deserialization=True
    )
    initial_count = main_store.index.ntotal
    print(f"   ✅ Loaded! Initial count: {initial_count} chunks")

    # 2-2. 추가할 벡터 스토어 순회하며 병합
    for path in ADD_STORE_PATH:
        if not os.path.exists(path):
            print(f"   ⚠️ Path not found, skipping: {path}")
            continue

        print(f"➕ Merging store from: {path}")
        try:
            # 추가할 벡터 스토어 로드
            other_store = FAISS.load_local(
                path,
                embedding_model,
                allow_dangerous_deserialization=True
            )

            # 병합 (Merge)
            main_store.merge_from(other_store)
            print(f"   ✅ Merged! (Added {other_store.index.ntotal} chunks)")

        except Exception as e:
            print(f"   ❌ Error merging {path}: {e}")

    # 3. 최종 저장
    final_count = main_store.index.ntotal
    print(f"\n💾 Saving final combined store to: {FINAL_STORE_PATH}")
    print(f"   📊 Total Chunks: {initial_count} -> {final_count}")

    main_store.save_local(FINAL_STORE_PATH)
    print("🎉 All done! Use 'final_academic_store' for your RAG.")

except Exception as e:
    print(f"❌ Critical Error: {e}")

### 🔠 BM25 Retriever 병합 (Combine & Re-build)


In [ ]:
import pickle
import os
from langchain_community.retrievers import BM25Retriever

# 경로 설정
BASE_PATH = data_path
ADD_PATH = "/content/gdrive/MyDrive/Colab Notebooks/NLP-private"

# A. 기존 BM25 파일 경로
BASE_BM25_PATH = os.path.join(BASE_PATH, "final_embedding_4/academic_bm25_retriever.pkl")

# B. 추가할 BM25 파일 경로 리스트
ADD_BM25_PATH = [
    os.path.join(ADD_PATH, "embeddings_additional/academic_bm25_retriever.pkl"),
]

# C. 최종 저장할 통합 BM25 경로
FINAL_BM25_PATH = os.path.join(BASE_PATH, "final_embedding_5/academic_bm25_retriever.pkl")

In [ ]:
def merge_bm25_retrievers(base_store_path, add_store_paths, save_path):
    all_docs = []

    # 1. 기존 BM25에서 문서 추출
    print(f"📖 Loading my BM25 from: {base_store_path}")
    try:
        with open(base_store_path, 'rb') as f:
            my_retriever = pickle.load(f)
            # BM25Retriever 객체 내부의 docs 속성에 문서들이 들어있습니다.
            if hasattr(my_retriever, 'docs'):
                all_docs.extend(my_retriever.docs)
                print(f"   ✅ Extracted {len(my_retriever.docs)} docs")
            else:
                print("   ❌ Error: This pickle is not a standard LangChain BM25Retriever.")
                return
    except Exception as e:
        print(f"   ❌ Error loading my file: {e}")
        return

    # 2. 추가할 BM25에서 문서 추출
    for path in add_store_paths:
        if not os.path.exists(path):
            print(f"   ⚠️ File not found, skipping: {path}")
            continue

        print(f"➕ Loading teammate BM25 from: {path}")
        try:
            with open(path, 'rb') as f:
                tm_retriever = pickle.load(f)
                if hasattr(tm_retriever, 'docs'):
                    all_docs.extend(tm_retriever.docs)
                    print(f"   ✅ Extracted {len(tm_retriever.docs)} docs")
        except Exception as e:
            print(f"   ❌ Error merging {path}: {e}")

    # 3. 통합 BM25 재생성 (Re-build)
    if not all_docs:
        print("❌ No documents found to merge.")
        return

    print(f"\n⚙️ Re-building BM25 Retriever with {len(all_docs)} total chunks...")
    # 다시 통계를 계산하여 인덱스를 만듭니다.
    combined_bm25 = BM25Retriever.from_documents(all_docs)
    combined_bm25.k = 4 # 기본 검색 개수 설정

    # 4. 저장
    # 저장 폴더가 없으면 생성
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    with open(save_path, 'wb') as f:
        pickle.dump(combined_bm25, f)

    print(f"💾 Saved combined BM25 to: {save_path}")
    print("🎉 BM25 Merge Complete!")

# 실행
merge_bm25_retrievers(BASE_BM25_PATH, ADD_BM25_PATH, FINAL_BM25_PATH)

### 📝 Final Metadata JSON 생성

embedding 폴더에 **필요한 파일들을 모두 옮겨두었는지 확인** 해주세요!

**불러온 경로가 metadata 파일 내에 그대로 저장**되는 점 참고해주세요!

---

⭐ **metadata 파일 내에 FAISS의 chunk 개수를 살려두고 싶으신 경우** ⭐
  
  `metadata 생성 코드` 실행 후, `🔎 실제 로딩된 데이터 개수 확인` 코드를 실행해주세요!
  
  이후, 출력 결과를 `metadata 생성 코드` 내 `num_chunks` 옆에 직접 작성하고 재실행 해주시면 됩니다...

In [ ]:
# @title metadata 생성 코드
import json
import os

# 1. 경로 설정
BASE_PATH = data_path

# A. EWHA 스토어 경로
EWHA_STORE_PATH = os.path.join(BASE_PATH, "final_embedding_5/ewha-vectorstore")
EWHA_BM25_PATH = os.path.join(BASE_PATH, "final_embedding_5/ewha_bm25_retriever.pkl")

# B. academic 스토어 경로
ACADEMIC_STORE_PATH = os.path.join(BASE_PATH, "final_embedding_5/academic-vectorstore")
ACADEMIC_BM25_PATH = os.path.join(BASE_PATH, "final_embedding_5/academic_bm25_retriever.pkl")

# C. 저장할 메타데이터 파일 경로
SAVE_JSON_PATH = os.path.join(BASE_PATH, "final_embedding_5/metadata.json")


# 2. 메타데이터 딕셔너리 생성
final_metadata = {
    "ewha": {
        "vectorstore_path": EWHA_STORE_PATH,
        "bm25_path": EWHA_BM25_PATH,
        "num_chunks": 41 # (선택사항) 필요하다면 로드해서 카운트 확인 가능
    },
    "academic": {
        "vectorstore_path": ACADEMIC_STORE_PATH,
        "bm25_path": ACADEMIC_BM25_PATH,
        "num_chunks": 28247 # (선택사항) 필요하다면 로드해서 카운트 확인 가능
    }
}

# 3. JSON 파일로 저장
try:
    with open(SAVE_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(final_metadata, f, indent=4, ensure_ascii=False)

    print(f"✅ New metadata file created at: {SAVE_JSON_PATH}")
    print("👉 Now update your run.py to load THIS file!")

except Exception as e:
    print(f"❌ Error creating metadata: {e}")

In [ ]:
# @title 🔎 실제 로딩된 데이터 개수 확인

# ===================================================================
# 저장된 임베딩 로드
# ===================================================================
embedding_save_path = os.path.join(data_path, "final_embedding_5")

# 메타데이터 로드
metadata_path = os.path.join(embedding_save_path, "metadata.json")
with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f"✓ 메타데이터 로드 완료")

# EWHA 임베딩 로드
print("\n【 EWHA 임베딩 로드 】")
ewha_embedding_path = metadata["ewha"]["vectorstore_path"]
ewha_vectorstore = FAISS.load_local(
    ewha_embedding_path,
    upstage_embeddings,
    allow_dangerous_deserialization=True
)

# 학문 자료 임베딩 로드
print("\n【 학문 자료 임베딩 로드 】")
academic_embedding_path = metadata["academic"]["vectorstore_path"]
academic_vectorstore = FAISS.load_local(
    academic_embedding_path,
    upstage_embeddings,
    allow_dangerous_deserialization=True
)

# -----------------------------
print(f"📊 [검증 결과]")

# 1. EWHA 스토어 확인
try:
    real_ewha_count = ewha_vectorstore.index.ntotal
    print(f"✅ EWHA 실제 청크 개수: {real_ewha_count}개 (정상)")
except Exception as e:
    print(f"❌ EWHA 스토어 오류: {e}")

# 2. Academic 스토어 확인
try:
    real_academic_count = academic_vectorstore.index.ntotal
    print(f"✅ Academic 실제 청크 개수: {real_academic_count}개 (정상)")
except Exception as e:
    print(f"❌ Academic 스토어 오류: {e}")

if real_ewha_count > 0 and real_academic_count > 0:
    print("\n🎉 결론: 데이터는 완벽하게 로드되었습니다. 0이라고 뜬 건 무시하고 진행하세요!")
    '''